# Chapter 22: Convolutional Neural Networks. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/22-cnn.py`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/22-cnn.py).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/22-cnn.ipynb)

In [ ]:
# Modules that the exercises of this notebook use: numpy, scipy, torch (all installed on Colab)

# The exercises read files from data/: download the datasets of the book once
import os
import urllib.request
import zipfile

if not os.path.exists("data"):
    urllib.request.urlretrieve("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip")
    zipfile.ZipFile("data.zip").extractall()
    os.remove("data.zip")

## Exercise 22.6: 2D Convolution from Scratch

Implement 2D convolution (valid, no padding, stride 1) using only numpy, then use it to find a vertical edge in a small synthetic image. Here `image` is the $6\times 6$ array whose first three columns are 0 and last three columns are 1, and `sobel_x` is the $3\times 3$ Sobel kernel with rows $(-1, 0, 1)$, $(-2, 0, 2)$ and $(-1, 0, 1)$.

In [ ]:
import numpy as np

image = np.array([
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1, 1],
], dtype=float)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=float)

a) Write a function `conv2d(image, kernel)` that slides the kernel over the image with stride 1 and no padding, at each position summing the elementwise product of kernel and patch (as the `conv()` helper does in Chapter 22). For an $H \times W$ image and a $k \times k$ kernel with stride $s$, give the general formula for the output height and width, and check that it matches the shape returned by your function on `image` and `sobel_x`.

b) Apply `conv2d` to `image` using `sobel_x`. Where is the response largest, and why is this the behavior you would expect from a vertical edge detector? Verify your output against `scipy.signal.correlate2d(image, sobel_x, mode="valid")`. Then compare with `scipy.signal.convolve2d(image, sobel_x, mode="valid")`, which rotates the kernel 180 degrees before sliding it, the textbook definition of convolution. Do the two scipy results agree here, and what does that say about `sobel_x`?

c) The kernel above has 9 weights plus 1 bias, reused at all 16 output positions. How many parameters would a fully connected layer need to map all 36 input pixels to the same 16 outputs? What does this comparison illustrate about parameter sharing in convolutional layers?

## Exercise 22.7: Max and Average Pooling from Scratch

Chapter 22 says that local max pooling can reduce sensitivity to some shifts but does not guarantee an unchanged prediction. This exercise implements $2\times 2$ pooling with a chosen stride in numpy and checks exactly how much invariance that is. The input `x` is the $2\times 6$ array whose two rows are both $(0, 1, 9, 0, 2, 0)$.

In [ ]:
import numpy as np
np.random.seed(22)

x = np.array([
    [0., 1., 9., 0., 2., 0.],
    [0., 1., 9., 0., 2., 0.],
])

a) Write a function `pool2d(x, reducer, size=2, stride=2)` that slides a size-by-size window over `x` with the given stride (no padding) and fills each output entry with `reducer` applied to that window. Use it to define `max_pool2d` and `avg_pool2d` (with `reducer` equal to `np.max` and `np.mean`), apply both to `x`, and check the results against `expected_max = [[1, 9, 2]]` and `expected_avg = [[0.5, 4.5, 1.0]]` with `np.allclose`.

b) Build `x_shift` by shifting `x` one pixel to the right (fill the new first column with zeros, and drop the last column). Pool `x_shift` with both functions and compare, entry by entry, to the part (a) results.

c) The two large activations in `x`, the values 9 and 2, sit at columns 2 and 4. Using the part (b) comparison, explain why max pooling reproduces both of them exactly after the shift while average pooling does not, and relate this to why max pooling, rather than average pooling, is usually credited with giving CNNs a degree of translation invariance.

## Exercise 22.8: Training and Shrinking a CNN on MNIST in PyTorch

Chapter 22 trains a PyTorch convolutional network on 8,000 MNIST images and reports 93% test accuracy, against 77% for the fully connected network. This exercise reproduces that pipeline, then shrinks the convolutional channels and measures what the shrinking costs. Use the same data and split as the chapter: load the MNIST test files with the chapter's `parse_images` and `parse_labels`, then reuse `numpy.random.RandomState(0)` to build `traini` (the first 8,000 shuffled indices) and `testi` (the remaining 2,000). The chapter's training code calls `torch.manual_seed(0)` before building the model; do the same before building each model here, so that your numbers are reproducible.

In [ ]:
import gzip, struct, array
import numpy as np
import numpy.random as npr
import torch
import torch.nn as nn
from torch.nn import functional as F

def parse_labels(filename):
    with gzip.open(filename, "rb") as fh:
        _ = struct.unpack(">II", fh.read(8))
        return np.array(array.array("B", fh.read()), dtype=np.uint8)

def parse_images(filename):
    with gzip.open(filename, "rb") as fh:
        _, num, rows, cols = struct.unpack(">IIII", fh.read(16))
        return np.array(array.array("B", fh.read()), dtype=np.uint8).reshape(num, rows, cols)

x = parse_images("data/t10k-images-idx3-ubyte.gz")
y = parse_labels("data/t10k-labels-idx1-ubyte.gz")

rng = npr.RandomState(0)
n = x.shape[0]
perm = rng.permutation(n)
traini, testi = perm[0:8000], perm[8000:n]

batch_size, num_epochs = 128, 5
num_train = len(traini)
num_batches = (num_train + batch_size - 1) // batch_size

a) Implement the chapter's `Net`: `conv1` mapping 1 to 32 channels and `conv2` mapping 32 to 64 channels (both 3×3 valid convolutions, each followed by ReLU), then a single 2×2 max pool, a flatten to 9216 features, `fc1` mapping 9216 to 128 (with ReLU), and `fc2` mapping 128 to 10 with a log-softmax output. Train for 5 epochs at batch size 128, minimizing `F.nll_loss` with `torch.optim.Adadelta` at `lr=0.1`. Print the training loss at the end of each epoch and report the final test accuracy on `testi`.

b) Build a smaller network identical to `Net` except that `conv1` outputs 16 channels and `conv2` outputs 32, so the flattened size becomes 32×12×12 = 4608 and `fc1` maps 4608 to 128. Train it with the same data split, batch size, epoch count, and optimizer settings. Report its loss trajectory and final test accuracy.

c) For both networks, compute the total parameter count with `sum(p.numel() for p in model.parameters())` and the count for `fc1` alone. What fraction of each network's parameters sits in `fc1`? By roughly what factor did halving the channel counts reduce the total?

d) Compare the two networks' accuracies and parameter counts, to each other and to the chapter's 77% fully connected baseline. Does halving the channels cost much accuracy? Using your answer to (c), explain why shrinking the convolutional channels cuts the total parameter count by roughly the same factor even though the convolutional layers themselves hold under 2% of the parameters, and connect this to Exercise 22.5's point about where a CNN's real parameter cost lives.